In [1]:
from hipporag import HippoRAG

/home/dzigen/miniconda3/envs/hipporag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-10 12:36:08,641	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:
# Prepare datasets and evaluation
docs = [
    "Oliver Badman is a politician.",
    "George Rankin is a politician.",
    "Thomas Marwick is a politician.",
    "Cinderella attended the royal ball.",
    "The prince used the lost glass slipper to search the kingdom.",
    "When the slipper fit perfectly, Cinderella was reunited with the prince.",
    "Erik Hort's birthplace is Montebello.",
    "Marina is bom in Minsk.",
    "Montebello is a part of Rockland County."
]

In [2]:
hipporag = HippoRAG(save_dir='outputs', 
    llm_model_name='qwen2.5:7b',
    llm_base_url='http://localhost:11437/v1',
    embedding_model_name = 'facebook/contriever')

In [7]:
hipporag.get_graph_info()

{'num_phrase_nodes': 16,
 'num_passage_nodes': 9,
 'num_total_nodes': 25,
 'num_extracted_triples': 11,
 'num_triples_with_passage_node': 0,
 'num_synonymy_triples': 11,
 'num_total_triples': 22}

In [5]:
hipporag.global_config.save_openie = False

In [6]:
#Run indexing
hipporag.index(docs=docs)

'NoneType' object has no attribute 'group'
'NoneType' object has no attribute 'group'
'NoneType' object has no attribute 'group'


[]


Extracting triples: 100%|██████████| 9/9 [00:00<00:00, 91.85it/s, total_prompt_tokens=4300, total_completion_tokens=594, num_cache_hit=9]
9it [00:00, 19701.85it/s]
9it [00:00, 35679.33it/s]


In [9]:
#Separate Retrieval & QA
queries = [
    "What is George Rankin's occupation?",
    "How did Cinderella reach her happy ending?",
    "What county is Erik Hort's birthplace a part of?"
]

In [10]:
retrieval_results = hipporag.retrieve(queries=queries, num_to_retrieve=2)

Retrieving: 100%|██████████| 3/3 [00:00<00:00, 474.81it/s]


In [11]:
retrieval_results

[QuerySolution(question="What is George Rankin's occupation?", docs=['George Rankin is a politician.', 'Thomas Marwick is a politician.'], doc_scores=array([0.10512545, 0.02640661]), answer=None, gold_answers=None, gold_docs=None),
 QuerySolution(question='How did Cinderella reach her happy ending?', docs=['The prince used the lost glass slipper to search the kingdom.', 'Cinderella attended the royal ball.'], doc_scores=array([0.04469707, 0.04208313]), answer=None, gold_answers=None, gold_docs=None),
 QuerySolution(question="What county is Erik Hort's birthplace a part of?", docs=["Erik Hort's birthplace is Montebello.", 'Montebello is a part of Rockland County.'], doc_scores=array([0.10276673, 0.05196531]), answer=None, gold_answers=None, gold_docs=None)]

In [ ]:
qa_results = hipporag.rag_qa(retrieval_results)

In [10]:
rag_results = hipporag.rag_qa(queries=queries)

QA Reading: 100%|██████████| 3/3 [00:04<00:00,  1.43s/it]
Extraction Answers from LLM Response: 3it [00:00, 15669.88it/s]


In [ ]:
rag_results

In [12]:
#For Evaluation
answers = [
    ["Politician"],
    ["By going to the ball."],
    ["Rockland County"]
]

gold_docs = [
    ["George Rankin is a politician."],
    ["Cinderella attended the royal ball.",
    "The prince used the lost glass slipper to search the kingdom.",
    "When the slipper fit perfectly, Cinderella was reunited with the prince."],
    ["Erik Hort's birthplace is Montebello.",
    "Montebello is a part of Rockland County."]
]

rag_results = hipporag.rag_qa(queries=queries, 
                              gold_docs=gold_docs,
                              gold_answers=answers)

Retrieving: 100%|██████████| 3/3 [00:00<00:00, 345.56it/s]
Length of retrieved docs (9) is smaller than largest topk for recall score (200)
Length of retrieved docs (9) is smaller than largest topk for recall score (200)
Length of retrieved docs (9) is smaller than largest topk for recall score (200)
QA Reading: 100%|██████████| 3/3 [00:00<00:00, 2126.57it/s]
Extraction Answers from LLM Response: 3it [00:00, 48960.75it/s]


In [13]:
rag_results

([QuerySolution(question="What is George Rankin's occupation?", docs=['George Rankin is a politician.', 'Thomas Marwick is a politician.', 'Oliver Badman is a politician.', "Erik Hort's birthplace is Montebello.", 'When the slipper fit perfectly, Cinderella was reunited with the prince.', 'Cinderella attended the royal ball.', 'Montebello is a part of Rockland County.', 'The prince used the lost glass slipper to search the kingdom.', 'Marina is bom in Minsk.'], doc_scores=array([0.10512545, 0.02640661, 0.02606118, 0.00294715, 0.00280574,
         0.00204492, 0.00101014, 0.00033695, 0.        ]), answer="George Rankin's occupation is a politician.", gold_answers=['Politician'], gold_docs=['George Rankin is a politician.']),
  QuerySolution(question='How did Cinderella reach her happy ending?', docs=['The prince used the lost glass slipper to search the kingdom.', 'Cinderella attended the royal ball.', 'When the slipper fit perfectly, Cinderella was reunited with the prince.', "Erik Hort